# Group 8 — ISIC skin-lesion classification

Coding component: one ResNet-50 classifier over the fixed seven-class label space
`MEL, NV, BCC, AKIEC, BKL, DF, VASC`, trained on ISIC 2018 (HAM10000) and evaluated on the internal
lesion-disjoint test split and on the ISIC 2019 external set.

Every cell runs in order from top to bottom, and every code cell verifies its own output with
assertions before the next cell consumes it.

**Datasets (both are used)**
- ISIC 2018 Task 3 (HAM10000, 7 classes) — train / validation / internal test, lesion-disjoint split
- ISIC 2019 (8 classes) — external validation, SCC excluded, hash-deduped against 2018

The third dataset listed in the client document (SIIM-ISIC 2020) is not used: its archive is over
100 GB. This deviation is recorded in `AGENTS.md` and stated in the report.

**Run record** — RUN_RECORD_PLACEHOLDER


## Requirement map

| Section | Requirement | Main artifact |
|---|---|---|
| 0 | setup | paths, output folders, dataset file counts |
| 1 | 2.1 | library version table |
| 2 | 2.2 | gradient tape demonstration |
| 3 | 2.3 | training vs evaluation memory |
| data | foundation | `isic2018` / `isic2019` tables |
| 5 | 2.4 | preprocessing and label encoding |
| 6 | 2.5–2.6 | augmentation figure |
| 7 | 2.7–2.9 | `model.summary()`, feature maps |
| 8 | 1.1 | gradient-magnitude figure, VGG vs ResNet |
| 9 | 2.10–2.13 | training history and learning curves |
| 10 | 2.14–2.17 | confusion matrix, top-3 confident errors |
| 11 | 2.18–2.20 | external results, unseen image, calibration |
| 12 | submission | saved artifacts and results archive |

Coverage checklist: 1.1 · 2.1 · 2.2 · 2.3 · 2.4 · 2.5 · 2.6 · 2.7 · 2.8 · 2.9 · 2.10 · 2.11 ·
2.12 · 2.13 · 2.14 · 2.15 · 2.16 · 2.17 · 2.18 · 2.19 · 2.20


## Run plan

- **Once, before running anything**: download the two Kaggle datasets and unpack them into `data/`
  (the exact folder names are listed in section 0).
- **Session 1**: sections 0 → data layer; the printed counts must be larger than zero and every
  assertion must pass.
- **Session 2**: sections 2.1 → 2.9 (cheap; the decoded image cache is built here once).
- **Session 3**: sections 2.10 → 2.13 — 100 training epochs (3 head + 97 fine-tuning), roughly ten
  hours on this CPU-only machine. Every epoch writes `outputs/model/last.weights.h5` and one row of
  `outputs/tables/train_history.csv`, so an interrupted run continues from the last completed epoch
  when this cell is executed again.
- **Session 4**: sections 2.14 → 2.20 (evaluation, external validation and figures).


## 0. Setup and runtime

Download the two datasets from Kaggle **once** and unpack them into this project like this:

```
data/
  ISIC 2018/   HAM10000_metadata.csv + *.jpg   (HAM10000, 7 classes, 10,015 images)
  ISIC 2019/   ISIC_2019_Training_GroundTruth.csv + *.jpg
```

`data/` and `outputs/` are git-ignored, so the images never enter the repository.
The last line of the cell prints how many JPEGs are visible per dataset; a wrong folder name shows up
immediately as `0 jpg`.

Produces: fixed paths, output folders, seeds, dataset file counts.


In [ ]:
# Section 0 - paths and seeds. The two datasets are downloaded once from Kaggle and unpacked here:
#   data/ISIC 2018/  data/ISIC 2019/
import os, random, hashlib
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # keep TensorFlow C++ logs out of the notebook output
from pathlib import Path
import numpy as np
import pandas as pd

def project_root(start: Path) -> Path:
    """Return the project root whether the kernel starts in the root or in code/ (local Jupyter)."""
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir() or (candidate / "AGENTS.md").is_file():
            return candidate
    raise RuntimeError("open this notebook inside the project folder (data/ not found)")

ROOT = project_root(Path.cwd())
os.chdir(ROOT)                # keep every relative path below stable and portable
DATA_ROOT = Path("data")      # project folder; keep the same two subfolders
OUT_DIR = Path("outputs")     # figures and tables produced by later sections
(OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]  # fixed 7-class label space
IMG_SIZE = 224
BATCH_SIZE = 32

SEED = 8
random.seed(SEED)
np.random.seed(SEED)

assert DATA_ROOT.is_dir(), "create data/ with the two dataset folders first"
# First sanity check: every line must show a number larger than 0.
for name in ("ISIC 2018", "ISIC 2019"):
    print(f"{name:10s} {len(list((DATA_ROOT / name).rglob('*.jpg'))):6d} jpg")


## 2.1 Libraries

Required: list every library used and its version. Produces: printed version table saved to `env.json`.

In [ ]:
# 2.1 - Record every library used and its version (saved to outputs/tables/env.json).
import warnings
warnings.filterwarnings("ignore")
import json, math, platform, time
import tensorflow as tf
import keras
import sklearn
import sklearn.metrics
from sklearn.model_selection import train_test_split
import PIL
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

# TF 2.21 / CPU: the MLIR meta optimizer prints spurious "operand does not dominate" errors for
# GradientTape graphs. Disabling it keeps the output clean and leaves training speed unchanged.
tf.config.optimizer.set_experimental_options({"disable_meta_optimizer": True})

ENV = {
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
    "pillow": PIL.__version__,
}
env_df = pd.DataFrame({"library": list(ENV), "version": list(ENV.values())})
print(env_df.to_string(index=False))
(OUT_DIR / "tables" / "env.json").write_text(json.dumps(ENV, indent=2))

# One seed for python, numpy and tensorflow; section 0 already fixed SEED.
tf.keras.utils.set_random_seed(SEED)
print("\nrandom seed:", SEED)


## 2.2 Forward pass and automatic differentiation

Required: explain how the framework tracks the forward pass and recovers gradients. Produces: per-layer gradient norms for one real batch.

**What the tape records.** While the `GradientTape` context is open, TensorFlow stores every operation whose inputs include a trainable variable together with the tensors needed to differentiate it. Closing the context freezes that recording; `tape.gradient()` then replays it in reverse (chain rule) and returns one gradient tensor per trainable variable. The tape keeps references to intermediate activations until it is consumed, which is why training needs more memory than inference.


In [ ]:
# 2.2 - A real image batch, one real forward pass, and the gradients the tape recovers.
def load_raw_batch(folder, n=8):
    """Read JPEGs straight from disk: this is the autodiff demo, full preprocessing comes in 2.4."""
    paths = sorted(p for p in (DATA_ROOT / folder).rglob("*.jpg"))[:n]
    batch = [np.asarray(Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32) / 255.0
             for p in paths]
    return tf.convert_to_tensor(np.stack(batch)), paths

demo_x, demo_paths = load_raw_batch("ISIC 2018")
demo_y = tf.one_hot([0, 1, 2, 3, 4, 5, 6, 0], depth=len(CLASS_NAMES))

demo_model = tf.keras.Sequential([
    tf.keras.layers.Input((IMG_SIZE, IMG_SIZE, 3)),
    tf.keras.layers.Conv2D(16, 3, strides=2, activation="relu"),
    tf.keras.layers.Conv2D(32, 3, strides=2, activation="relu"),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(len(CLASS_NAMES)),
], name="tape_demo")
demo_loss = tf.keras.losses.CategoricalCrossentropy(from_logits=True)

# Every operation that depends on a trainable variable is recorded while the context is open.
with tf.GradientTape() as tape:
    logits = demo_model(demo_x, training=True)
    loss = demo_loss(demo_y, logits)

# gradient() replays the recorded graph backwards (chain rule) and returns one tensor per variable.
grads = tape.gradient(loss, demo_model.trainable_variables)
print("demo batch:", tuple(demo_x.shape), "| loss:", round(float(loss), 4))

grad_rows = [{"layer": var.name, "shape": tuple(var.shape),
              "grad_norm": float(tf.norm(g)) if g is not None else 0.0}
             for var, g in zip(demo_model.trainable_variables, grads)]
grad_df = pd.DataFrame(grad_rows)
print(grad_df.to_string(index=False))
grad_df.to_csv(OUT_DIR / "tables" / "gradient_norms_2_2.csv", index=False)


## 2.3 Training and evaluation memory

Required: explain the memory difference between training and evaluation mode. Produces: behaviour evidence and written comparison.

**Why training costs more memory.** In training mode the framework keeps every intermediate activation needed by the backward pass, BatchNormalization uses the statistics of the current batch, and dropout samples a new mask on every call. In evaluation mode no backward pass follows, so activations can be released as soon as they are produced, BatchNormalization reuses the moving statistics learned during training, and dropout is switched off - the output is deterministic and no tape is needed. The printed comparison shows the two regimes: training-mode calls disagree with each other, evaluation-mode calls are identical.


In [ ]:
# 2.3 - Training mode versus evaluation mode: behaviour evidence and cost evidence.
demo_model3 = tf.keras.Sequential([
    tf.keras.layers.Input((IMG_SIZE, IMG_SIZE, 3)),
    tf.keras.layers.Conv2D(16, 3, strides=2, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(len(CLASS_NAMES)),
], name="mode_demo")

train_out_a = demo_model3(demo_x, training=True)
train_out_b = demo_model3(demo_x, training=True)
eval_out_a = demo_model3(demo_x, training=False)
eval_out_b = demo_model3(demo_x, training=False)
print("training=True  -> max |call1 - call2|:", round(float(tf.reduce_max(tf.abs(train_out_a - train_out_b))), 6))
print("training=False -> max |call1 - call2|:", round(float(tf.reduce_max(tf.abs(eval_out_a - eval_out_b))), 6))

@tf.function
def forward_eval(x):
    return demo_model3(x, training=False)

@tf.function
def forward_train_with_tape(x):
    with tf.GradientTape() as tape:
        y = demo_model3(x, training=True)
        loss = tf.reduce_mean(y)
    return tape.gradient(loss, demo_model3.trainable_variables)

def time_forward(fn, repeats=3):
    fn(demo_x)                      # first call traces the tf.function
    start = time.perf_counter()
    for _ in range(repeats):
        fn(demo_x)
    return (time.perf_counter() - start) / repeats

t_eval = time_forward(forward_eval)
t_train = time_forward(forward_train_with_tape)
print(f"eval forward: {t_eval * 1000:.1f} ms | training forward + backward: {t_train * 1000:.1f} ms")
print(f"one training step costs {t_train / t_eval:.1f}x the evaluation pass on this batch")


## Data layer: load the two datasets from the project folder

No download code is needed: the folders above are read directly. Each dataset becomes one small table
with the columns the later sections need (`image`, `path`, `label` / `target`, `split`, `md5`).

Rules applied here:
- fixed 7-class label space; HAM10000 lowercase codes are mapped to class names;
- lesion-level split for ISIC 2018, so all images of one lesion stay in one split;
- SCC dropped from ISIC 2019 (the model has no SCC slot);
- duplicate images removed from ISIC 2019 by file md5 (it largely repeats ISIC 2018).

Produces: the `isic2018` and `isic2019` tables, plus `outputs/tables/isic2018_split.csv`.


In [ ]:
# 2.4 - ISIC 2018 (HAM10000): build image path + 7-class label + lesion id.
# HAM10000 stores lowercase codes such as "mel"; map them to the fixed class names.
DX_TO_CLASS = {"mel": "MEL", "nv": "NV", "bcc": "BCC", "akiec": "AKIEC",
               "bkl": "BKL", "df": "DF", "vasc": "VASC"}
EXPECTED_2018_CLASS_COUNTS = {"MEL": 1113, "NV": 6705, "BCC": 514, "AKIEC": 327,
                              "BKL": 1099, "DF": 115, "VASC": 142}

base = DATA_ROOT / "ISIC 2018"
meta_path = next(base.rglob("HAM10000_metadata.csv"), None)
assert meta_path is not None, "data/ISIC 2018 must contain HAM10000_metadata.csv"
meta = pd.read_csv(meta_path)
jpgs = {p.stem: p for p in base.rglob("*.jpg")}     # image name -> full path

isic2018 = meta[["image_id", "lesion_id", "dx"]].rename(columns={"image_id": "image"})
isic2018["label"] = isic2018["dx"].map(DX_TO_CLASS)
isic2018["path"] = isic2018["image"].map(lambda name: str(jpgs[name]))
print(isic2018["label"].value_counts().to_string())

# --- verification: the table must describe the complete HAM10000 collection exactly once.
assert len(isic2018) == 10015, f"expected 10015 HAM10000 rows, found {len(isic2018)}"
assert isic2018["image"].is_unique, "duplicate image identifiers"
assert isic2018["label"].notna().all(), "an unknown dx code was mapped to NaN"
assert isic2018["label"].nunique() == len(CLASS_NAMES)
assert isic2018["lesion_id"].nunique() == 7470, "HAM10000 contains 7470 lesions"
assert isic2018.groupby("lesion_id")["label"].nunique().max() == 1, "one lesion has two diagnoses"
assert (isic2018["label"].value_counts().reindex(CLASS_NAMES).tolist()
        == list(EXPECTED_2018_CLASS_COUNTS.values())), "class counts differ from the official release"
missing_files = [p for p in isic2018["path"] if not Path(p).is_file() or Path(p).stat().st_size == 0]
assert not missing_files, f"{len(missing_files)} images are missing, first: {missing_files[:3]}"
assert len(list(base.rglob("*.jpg"))) >= len(isic2018), "more JPEGs than metadata rows: check data/ layout"
print(f"ISIC 2018 verified: {len(isic2018)} images, {isic2018['lesion_id'].nunique()} lesions")


In [ ]:
# Lesion-level split: all images of one lesion land in the same split (no leakage between splits).
# The split is stratified by diagnosis, so every split keeps the class proportions of HAM10000.
lesions = (isic2018[["lesion_id", "label"]].drop_duplicates()
           .sort_values("lesion_id").reset_index(drop=True))
assert len(lesions) == 7470 and lesions["lesion_id"].is_unique

trainval_lesions, test_lesions = train_test_split(
    lesions, test_size=0.15, stratify=lesions["label"], random_state=SEED)
train_lesions, val_lesions = train_test_split(
    trainval_lesions, test_size=0.15 / 0.85, stratify=trainval_lesions["label"], random_state=SEED)

split_by_lesion = {lesion: "train" for lesion in train_lesions["lesion_id"]}
split_by_lesion.update({lesion: "validation" for lesion in val_lesions["lesion_id"]})
split_by_lesion.update({lesion: "test" for lesion in test_lesions["lesion_id"]})
isic2018["split"] = isic2018["lesion_id"].map(split_by_lesion)
print(isic2018["split"].value_counts().to_string())

# --- verification: expected sizes, class coverage and zero leakage between the three splits.
split_frames = {name: isic2018[isic2018["split"] == name] for name in ("train", "validation", "test")}
image_counts = [len(frame) for frame in split_frames.values()]
lesion_counts = [frame["lesion_id"].nunique() for frame in split_frames.values()]
assert image_counts == [7028, 1488, 1499], f"unexpected image split sizes {image_counts}"
assert lesion_counts == [5228, 1121, 1121], f"unexpected lesion split sizes {lesion_counts}"
assert sum(image_counts) == 10015 and sum(lesion_counts) == 7470, "split does not cover the dataset"
for name, frame in split_frames.items():
    assert set(frame["label"]) == set(CLASS_NAMES), f"{name} split misses a diagnosis class"
for first, second in (("train", "validation"), ("train", "test"), ("validation", "test")):
    assert set(split_frames[first]["lesion_id"]).isdisjoint(split_frames[second]["lesion_id"]), \
        f"{first} and {second} share a lesion"
    assert set(split_frames[first]["image"]).isdisjoint(split_frames[second]["image"]), \
        f"{first} and {second} share an image"
print("split sizes:", dict(zip(split_frames, image_counts)))

isic2018.to_csv(OUT_DIR / "tables" / "isic2018_split.csv", index=False)
(OUT_DIR / "tables" / "data_provenance.json").write_text(json.dumps({
    "dataset": "ISIC 2018 Task 3 / HAM10000",
    "external_dataset": "ISIC 2019 training set (8 classes, SCC excluded for the 7-class mapping)",
    "source": "Kaggle archives unpacked manually into data/ (the notebook contains no download code)",
    "license": "CC BY-NC 4.0",
    "seed": SEED,
    "split_unit": "lesion_id",
    "split_protocol": "lesion-level 70/15/15, stratified by diagnosis",
    "split_sizes": dict(zip(split_frames, image_counts)),
}, indent=2))


In [ ]:
# 2.4 - ISIC 2019 is the external validation set. It has 8 classes; SCC has no slot in our 7 classes.
# The ground-truth CSV stores AKIEC as "AK", so the column names are mapped to the fixed class names.
ISIC2019_CLASSES = {"MEL": "MEL", "NV": "NV", "BCC": "BCC", "AK": "AKIEC",
                    "BKL": "BKL", "DF": "DF", "VASC": "VASC", "SCC": "SCC"}

base = DATA_ROOT / "ISIC 2019"
jpgs = {p.stem: p for p in base.rglob("*.jpg")}
truth_path = next(base.rglob("ISIC_2019_Training_GroundTruth.csv"), None)
assert truth_path is not None, "data/ISIC 2019 must contain ISIC_2019_Training_GroundTruth.csv"
truth = pd.read_csv(truth_path)

# --- verification: the ground truth is one-hot, and the 7-class mapping is complete.
one_hot = truth[list(ISIC2019_CLASSES)].to_numpy()
assert one_hot.shape == (25331, 8), f"unexpected ISIC 2019 ground-truth shape {one_hot.shape}"
assert (one_hot.sum(axis=1) == 1).all(), "ISIC 2019 ground truth is not one-hot"
truth["label"] = truth[list(ISIC2019_CLASSES)].idxmax(axis=1).map(ISIC2019_CLASSES)
assert truth["label"].notna().all()
scc_count = int((truth["label"] == "SCC").sum())
assert scc_count == 628, f"unexpected SCC count {scc_count}"

isic2019 = truth[truth["label"] != "SCC"][["image", "label"]].copy()
isic2019["path"] = isic2019["image"].map(lambda name: str(jpgs[name]))
isic2019["split"] = "external_val"
assert len(isic2019) == 24703 and isic2019["label"].nunique() == 7
missing_files = [p for p in isic2019["path"] if not Path(p).is_file() or Path(p).stat().st_size == 0]
assert not missing_files, f"{len(missing_files)} ISIC 2019 images are missing, first: {missing_files[:3]}"
print("isic2019:", len(isic2019), "| SCC dropped:", scc_count)


In [ ]:
# ISIC 2019 largely repeats ISIC 2018 images, so remove them: the external set must be unseen.
def md5(path):
    with open(path, "rb") as handle:
        return hashlib.md5(handle.read()).hexdigest()

isic2018["md5"] = [md5(p) for p in isic2018["path"]]
isic2019["md5"] = [md5(p) for p in isic2019["path"]]
duplicates = isic2019["md5"].isin(isic2018["md5"])
isic2019 = isic2019[~duplicates].reset_index(drop=True)

# --- verification: the external split shares no image with ISIC 2018 and keeps all seven classes.
assert len(isic2019) == 14885, f"unexpected external size {len(isic2019)}"
assert not isic2019["md5"].isin(isic2018["md5"]).any(), "external set still overlaps ISIC 2018"
print("repeated images left inside isic2019:", int(isic2019["md5"].duplicated().sum()))
assert isic2019["label"].nunique() == 7 and not isic2019["path"].isna().any()
print("duplicate images removed from isic2019:", int(duplicates.sum()))
print("rows -> isic2018:", len(isic2018), "| isic2019:", len(isic2019))


## 2.4 Preprocessing and label encoding

Required: preprocessing, augmentation and label encoding. Produces: fixed class order, uint8 cache, `tf.data` pipelines, one-hot labels.

**Preprocessing here.** Each JPEG is decoded and resized to `IMG_SIZE` once and cached as `uint8` under `outputs/cache/`, so later sessions skip JPEG decoding; the cache file name contains a hash of the split rows, the image size and the row count, so a stale cache can never be reused after the seed or the split changes. The pipelines keep the pixels at 0-255 and only add the one-hot label: the ImageNet normalisation and the training augmentation are layers inside the model (section 2.7), so training, validation, test and single-image inference all pass through exactly the same preprocessing code. Labels use the fixed `CLASS_NAMES` order, the three `tf.data` pipelines shuffle (train only), batch and prefetch, and the label encoding is checked against the split table.


In [ ]:
# 2.4 - Preprocessing and label encoding: decode/resize once, cache uint8 arrays, build tf.data pipelines.
CACHE_DIR = OUT_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
AUTOTUNE = tf.data.AUTOTUNE
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}

def decode_resize(path):
    """JPEG bytes -> uint8 RGB tensor of shape (IMG_SIZE, IMG_SIZE, 3)."""
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE), method="bilinear")
    return tf.cast(image, tf.uint8)

def load_split(split_df):
    """Decode one split with a parallel tf.data map; return uint8 images and integer labels."""
    paths = tf.constant(split_df["path"].tolist())
    labels = np.array([CLASS_TO_INDEX[label] for label in split_df["label"]], dtype=np.int32)
    images = tf.data.Dataset.from_tensor_slices(paths).map(decode_resize, num_parallel_calls=AUTOTUNE)
    return np.stack(list(images.as_numpy_iterator())), labels

def cache_key(split_df, name):
    """Name the cache after the exact rows, image size and row count it was built from, so a stale
    cache can never be reused after the seed, the split protocol or IMG_SIZE changes."""
    digest = hashlib.md5("|".join(sorted(split_df["image"])).encode()).hexdigest()[:12]
    return f"{name}_{IMG_SIZE}_{len(split_df)}_{digest}"

SPLITS = {}
for name in ("train", "validation", "test"):
    split_df = isic2018[isic2018["split"] == name]
    key = cache_key(split_df, name)
    x_path, y_path = CACHE_DIR / f"{key}_x.npy", CACHE_DIR / f"{key}_y.npy"
    if x_path.is_file() and y_path.is_file():
        x, y, source = np.load(x_path), np.load(y_path), "reused cache " + key
    else:
        start = time.perf_counter()
        x, y = load_split(split_df)
        np.save(x_path, x)
        np.save(y_path, y)
        source = f"decoded in {time.perf_counter() - start:.1f}s -> {key}"
    # --- verification: the cache must describe exactly this split at this image size.
    assert x.shape[1:] == (IMG_SIZE, IMG_SIZE, 3) and x.dtype == np.uint8, "cache does not match IMG_SIZE"
    assert len(x) == len(y) == len(split_df), "cache row count does not match the split"
    assert (y == np.array([CLASS_TO_INDEX[label] for label in split_df["label"]])).all(), "cache labels are misaligned"
    assert np.array_equal(np.unique(y), np.arange(len(CLASS_NAMES))), "a split misses a class"
    SPLITS[name] = (x, y)
    print(f"{name:10s} {x.shape[0]:5d} images | {x.shape[1]}x{x.shape[2]}x{x.shape[3]} | {x.dtype} | {source}")

def prepare_sample(image, label):
    """Keep the pixels at 0-255 and one-hot encode the label: the model applies the augmentation and
    the ImageNet preprocessing itself, so a [0, 1] round trip here would only lose precision."""
    return tf.cast(image, tf.float32), tf.one_hot(label, len(CLASS_NAMES))

x_train, y_train = SPLITS["train"]
x_val, y_val = SPLITS["validation"]
x_test, y_test = SPLITS["test"]

train_base = tf.data.Dataset.from_tensor_slices((x_train, y_train)).map(prepare_sample, num_parallel_calls=AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val)).map(prepare_sample, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).map(prepare_sample, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_labels = y_train.copy()          # integer labels kept for the class counts below
class_counts = pd.Series(train_labels).map(CLASS_NAMES.__getitem__).value_counts().reindex(CLASS_NAMES)
print("\ntraining images per class:\n" + class_counts.to_string())
print(f"\ntrain batches per epoch: {math.ceil(len(train_labels) / BATCH_SIZE)}")
for images, labels in val_ds.take(1):
    print("validation batch:", tuple(images.shape), "| labels:", tuple(labels.shape),
          "| pixel range:", (float(tf.reduce_min(images)), float(tf.reduce_max(images))),
          "| one-hot row:", labels[0].numpy().astype(int))
    assert images.shape[1:] == (IMG_SIZE, IMG_SIZE, 3) and labels.shape[1] == len(CLASS_NAMES)
    assert float(tf.reduce_max(images)) > 1.5, "pixels must stay at 0-255, the model does the normalisation"

# The datasets now hold the image tensors; release the numpy copies.
del SPLITS, x_train, y_train, x_val, y_val, x_test, y_test


## 2.5–2.6 Augmentation

Required: show two random augmentations of one sample side by side; explain variance without label change. Produces: augmentation figure.

**Augmentation.** Horizontal/vertical flips and small rotations are applied to training images only. The layer block sits inside the model (section 2.7), so it follows the model's training flag: active during training, skipped for validation, test and single-image inference. The transforms change the pixel statistics of an image - so the model sees extra variance - but never change its diagnosis label. The cell below prints the training batches per epoch and asserts that two consecutive random draws of the same sample are actually different, which would fail if the augmentation block had silently become a no-op.


In [ ]:
# 2.5 - Training-only augmentation. The layer block is part of the model (2.7), so it is applied only
# while the model is in training mode and is skipped for validation, test and single-image inference.
AUGMENT = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    tf.keras.layers.RandomRotation(0.10, seed=SEED),
], name="augment")

train_ds = train_base.shuffle(len(train_labels), seed=SEED).batch(BATCH_SIZE).prefetch(AUTOTUNE)

# 2.6 - One training sample and two independent random augmentations of the same sample.
first_train = isic2018[isic2018["split"] == "train"].iloc[0]
sample = tf.convert_to_tensor(
    np.asarray(Image.open(first_train["path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32))
batch = sample[None]
panels = [("original", sample),
          ("augmentation 1", AUGMENT(batch, training=True)[0]),
          ("augmentation 2", AUGMENT(batch, training=True)[0])]
for title, panel in panels:
    assert panel.shape == sample.shape, f"{title} changed the tensor shape"
    assert float(tf.reduce_min(panel)) >= 0.0 and float(tf.reduce_max(panel)) <= 255.0, f"{title} left the pixel range"
assert not np.allclose(panels[1][1].numpy(), panels[2][1].numpy()), "augmentation is not random"
assert not np.allclose(panels[0][1].numpy(), panels[1][1].numpy()), "augmentation did not change the image"

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, (title, panel) in zip(axes, panels):
    axis.imshow(np.clip(panel.numpy(), 0, 255).astype(np.uint8))
    axis.set_title(title)
    axis.axis("off")
fig.suptitle("2.5-2.6  same " + first_train["label"] + " sample, two random augmentations (label unchanged)",
             fontsize=11)
fig.tight_layout()
fig.savefig(OUT_DIR / "figures" / "augmentation_examples.png", dpi=150, bbox_inches="tight")
plt.show()
print("training batches per epoch:", math.ceil(len(train_labels) / BATCH_SIZE),
      "| augmentation layers:", [layer.name for layer in AUGMENT.layers])


## 2.7–2.9 Model, summary and feature maps

Required: build the model, print summary, extract feature maps, explain shapes and backpropagation. Produces: summary file, parameter counts, feature-map grid.

**Graph.** The inputs are 0-255 RGB pixels. The graph applies the training augmentation, converts the pixels to the caffe-style format the ImageNet weights expect (RGB -> BGR, per-channel mean subtraction), runs the frozen ResNet-50 in inference mode so its BatchNorm statistics never change, and finally pools, drops out and classifies. Because augmentation and normalisation are layers of the graph, every later call with `training=False` - validation, test and single-image inference - automatically reuses the same preprocessing without augmentation.

**Reading the model.** ResNet-50 first shrinks the spatial size (224 -> 7) while growing the channel depth (3 -> 2048), so the GAP layer sees one 2048-d vector per image; the dense head then maps it to 7 probabilities. Backpropagation starts from the softmax/cross-entropy gradient and applies the chain rule layer by layer, updating the head first and, in stage 2, the residual blocks. Residual shortcuts let the gradient flow around blocks instead of only through them.


In [ ]:
# 2.7 - ResNet-50 backbone (ImageNet weights) + GAP + dropout + softmax head.
# Re-seed here: the head initialisation then does not depend on how many demo models 2.2 and 2.3 built.
tf.keras.utils.set_random_seed(SEED)
# Augmentation and ImageNet preprocessing are part of the graph, so every inference path applies exactly
# the same preprocessing and the augmentation is skipped whenever the model is not training.
base_model = tf.keras.applications.ResNet50(include_top=False, weights="imagenet",
                                             input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False      # stage 1 trains the head only; 2.10 unfreezes the last stages
inputs = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3), dtype="float32", name="rgb_image")
x = AUGMENT(inputs)
x = tf.keras.layers.Lambda(tf.keras.applications.resnet50.preprocess_input,
                           dtype="float32", name="imagenet_preprocess")(x)
x = base_model(x, training=False)      # inference mode: the ImageNet BatchNorm statistics stay fixed
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pool")(x)
x = tf.keras.layers.Dropout(0.30, name="classifier_dropout")(x)
outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax", name="predictions")(x)
model = tf.keras.Model(inputs, outputs, name="resnet50_isic2018")

summary_lines = []
model.summary(print_fn=summary_lines.append)
summary_text = "\n".join(summary_lines)
(OUT_DIR / "tables" / "model_summary.txt").write_text(summary_text + "\n")
print(summary_text)
trainable_now = sum(int(np.prod(weight.shape)) for weight in model.trainable_weights)
print(f"\ntotal parameters: {model.count_params():,} | trainable in stage 1: {trainable_now:,}")

# 2.7 - Verification: the graph, the frozen backbone and the parameter counts must match the plan.
assert model.input_shape == (None, IMG_SIZE, IMG_SIZE, 3) and model.output_shape == (None, len(CLASS_NAMES))
assert not base_model.trainable_variables, "the backbone must be frozen before the head stage"
assert trainable_now == 14_343, f"unexpected number of trainable parameters {trainable_now:,}"
assert model.count_params() == 23_602_055, f"unexpected parameter count {model.count_params():,}"

# 2.8 - Feature maps from one hidden layer for one test image.
for images, labels in test_ds.take(1):
    test_image, test_label = images[0], int(tf.argmax(labels[0]))
feature_layer = base_model.get_layer("conv4_block6_out")
feature_extractor = tf.keras.Model(inputs=base_model.input, outputs=feature_layer.output)
preprocessed = tf.keras.applications.resnet50.preprocess_input(
    np.expand_dims(test_image.numpy(), axis=0))      # test_ds already yields 0-255 pixels
feature_maps = feature_extractor(tf.convert_to_tensor(preprocessed), training=False)[0].numpy()
print(f"feature map shape: {feature_maps.shape} | test image class: {CLASS_NAMES[test_label]}")
assert feature_maps.ndim == 3 and feature_maps.shape[0] == feature_maps.shape[1]

channels = np.linspace(0, feature_maps.shape[-1] - 1, 16).astype(int)
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, channel in zip(axes.ravel(), channels):
    ax.imshow(feature_maps[:, :, channel], cmap="viridis")
    ax.set_title(f"channel {channel}", fontsize=8)
    ax.axis("off")
fig.suptitle(f"2.8  feature maps at conv4_block6_out ({feature_maps.shape[0]}x{feature_maps.shape[1]}x{feature_maps.shape[2]})",
             fontsize=11)
fig.tight_layout()
fig.savefig(OUT_DIR / "figures" / "feature_maps.png", dpi=150, bbox_inches="tight")
plt.show()

# 2.9 - Why the spatial size shrinks while the channel depth grows (report section 2.9).
print("Input %dx%dx3 -> conv4_block6_out %dx%dx%d: spatial size / %d, channel depth x%.0f"
      % (IMG_SIZE, IMG_SIZE, feature_maps.shape[0], feature_maps.shape[1], feature_maps.shape[2],
         IMG_SIZE // feature_maps.shape[0], feature_maps.shape[2] / 3))


## Part 1.1 support: gradient magnitude per layer, VGG vs ResNet

Required by 1.1: a graph of gradient magnitudes per layer for VGG vs ResNet. Produces: the Part 1 figure and its data table.

**Vanishing gradients in a stack.** The chain rule multiplies the Jacobian of every layer, so repeated factors below 1 shrink the gradient as it travels backwards; in a plain VGG stack the earliest convolutions receive much smaller gradients than the last ones. ReLU keeps the derivative at 0 or 1 and therefore does not attenuate active paths the way saturating sigmoid/tanh units do, but it also switches off paths whose pre-activation is negative. Residual shortcuts add an identity term to the Jacobian, so part of the gradient reaches the early layers without being multiplied by every weight matrix. The figure records the L2 norm of the loss gradient for every trainable weight tensor of VGG16 and ResNet-50 on the same real batch.


In [ ]:
# 1.1 - Gradient magnitude per trainable layer for VGG16 versus ResNet-50, same real batch.
def imagenet_model(builder, preprocess_input, name):
    """Backbone with the ImageNet preprocessing it expects, plus a fresh 7-class head."""
    return tf.keras.Sequential([
        tf.keras.layers.Input((IMG_SIZE, IMG_SIZE, 3)),
        tf.keras.layers.Lambda(lambda x: preprocess_input(x * 255.0)),
        builder(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(len(CLASS_NAMES)),
    ], name=name)

def per_layer_gradient_table(net, batch):
    """L2 norm of d(loss)/d(weight) for every conv/dense kernel of one forward+backward pass."""
    labels = tf.one_hot([index % len(CLASS_NAMES) for index in range(batch.shape[0])], len(CLASS_NAMES))
    with tf.GradientTape() as tape:
        loss = tf.keras.losses.CategoricalCrossentropy()(labels, net(batch, training=True))
    gradients = tape.gradient(loss, net.trainable_variables)
    rows = []
    for depth, (variable, gradient) in enumerate(zip(net.trainable_variables, gradients)):
        if gradient is None or len(variable.shape) < 2:      # kernels only, skip biases
            continue
        rows.append({"depth_index": depth, "layer": variable.name,
                     "grad_norm": float(tf.norm(gradient))})
    return pd.DataFrame(rows)

vgg_net = imagenet_model(tf.keras.applications.VGG16, tf.keras.applications.vgg16.preprocess_input, "vgg16_gradients")
resnet_net = imagenet_model(tf.keras.applications.ResNet50, tf.keras.applications.resnet50.preprocess_input, "resnet50_gradients")
vgg_table = per_layer_gradient_table(vgg_net, demo_x).assign(model="VGG16")
resnet_table = per_layer_gradient_table(resnet_net, demo_x).assign(model="ResNet-50")
gradient_table = pd.concat([vgg_table, resnet_table], ignore_index=True)
gradient_table["relative_depth"] = gradient_table.groupby("model")["depth_index"].transform(
    lambda column: column / max(column.max(), 1))
gradient_table.to_csv(OUT_DIR / "tables" / "gradient_magnitudes_1_1.csv", index=False)
print(gradient_table.groupby("model")["grad_norm"].agg(["min", "median", "max"]).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
for model_name, color in (("VGG16", "tab:blue"), ("ResNet-50", "tab:orange")):
    part = gradient_table[gradient_table["model"] == model_name].sort_values("relative_depth")
    ax.plot(part["relative_depth"], part["grad_norm"].clip(lower=1e-12),
            color=color, marker="o", markersize=3, linewidth=1.2, label=model_name)
ax.set_yscale("log")
ax.set_xlabel("relative depth (input side -> output side)")
ax.set_ylabel("gradient L2 norm (log scale)")
ax.set_title("1.1  gradient magnitude per layer: VGG16 vs ResNet-50")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "figures" / "gradient_magnitudes_vgg_vs_resnet.png", dpi=150, bbox_inches="tight")
plt.show()

del vgg_net, resnet_net      # the gradient tables and figure are enough; free the two extra models


## 2.10–2.11 Training plan and learning-curve evidence

Required: adaptive optimizer, fixed epochs, loss curves with learning rate or gradient norm, overfitting analysis. Produces: training history and curves.

**Training plan.** The head is trained first for three epochs with the frozen ImageNet backbone (Adam, `1e-3`). The last two residual stages (`conv4`, `conv5`) are then unfrozen for 97 epochs, so the run uses exactly 100 epochs. `BatchNorm` layers stay frozen during fine-tuning: their moving statistics would otherwise be re-estimated from 32-image CPU batches and would drift away from the ImageNet values the convolution filters were adapted to. The fine-tuning rate follows a cosine schedule from `3e-5` to `1e-6`, which keeps the late epochs small enough to settle instead of oscillating. The loss is plain one-hot categorical cross-entropy: the class imbalance is measured and analysed in sections 2.14–2.17 instead of being hidden inside a reweighted loss. Every epoch saves `last.weights.h5` and appends one row to `outputs/tables/train_history.csv` (epoch, learning rate, training loss, training accuracy, validation loss, validation accuracy, validation macro F1, mean gradient norm, seconds), so an interrupted run resumes after the last completed epoch. `Adam` is the adaptive optimizer: it keeps first and second moment estimates per parameter and normalises each step by them, while the learning-rate schedule controls the overall step size.

**Gradient norm.** The mean gradient norm is recorded per epoch and printed below the curves. While the backbone is frozen only the head is trainable, so the norm is small; after `conv4`/`conv5` are unfrozen many more tensors receive gradients, so the norm rises. This is the same quantity that section 1.1 plots per layer.


In [ ]:
# 2.10 - Fixed training plan: three head epochs, then 97 fine-tuning epochs (100 epochs in total).
# Every epoch writes a checkpoint and one history row, so an interrupted run resumes automatically.
MODEL_DIR = OUT_DIR / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LAST_WEIGHTS = MODEL_DIR / "last.weights.h5"
BEST_WEIGHTS = MODEL_DIR / "best.weights.h5"
HISTORY_PATH = OUT_DIR / "tables" / "train_history.csv"

STAGE_A_EPOCHS = 3           # frozen backbone: train the classification head only
STAGE_B_EPOCHS = 97          # unfreeze the last two residual stages (100 epochs in total)
TOTAL_EPOCHS = STAGE_A_EPOCHS + STAGE_B_EPOCHS
HEAD_LR = 1e-3
FINETUNE_LR_START = 3e-5
FINETUNE_LR_END = 1e-6
HISTORY_DTYPES = {"stage": "object", "epoch": "int64", "lr": "float64", "train_loss": "float64",
                  "train_acc": "float64", "val_loss": "float64", "val_acc": "float64",
                  "val_macro_f1": "float64", "grad_norm": "float64", "seconds": "float64"}

EXPERIMENT_CONFIG = {
    "architecture": "ResNet-50, ImageNet weights, global average pooling + dropout 0.30 + softmax",
    "dataset": "ISIC 2018 / HAM10000 (7 classes), lesion-level stratified 70/15/15 split, seed 8",
    "head_epochs": STAGE_A_EPOCHS, "fine_tune_epochs": STAGE_B_EPOCHS, "total_epochs": TOTAL_EPOCHS,
    "head_lr": HEAD_LR, "fine_tune_lr": [FINETUNE_LR_START, FINETUNE_LR_END],
    "fine_tune_scope": "conv4/conv5 trainable, BatchNorm frozen",
    "optimizer": "Adam",
    "batch_size": BATCH_SIZE, "image_size": IMG_SIZE, "dropout": 0.30,
    "augmentation": ["RandomFlip horizontal+vertical", "RandomRotation 0.10"],
    "loss": "plain one-hot categorical cross-entropy (no class reweighting)",
    "imbalance_handling": "none in the loss; the class imbalance is measured in sections 2.14-2.17",
    "selection": "maximum validation accuracy, earlier epoch wins an exact tie",
    "evaluation": "internal lesion-disjoint test split, plus ISIC 2019 external validation"}
(OUT_DIR / "tables" / "experiment_config.json").write_text(json.dumps(EXPERIMENT_CONFIG, indent=2))
print(json.dumps(EXPERIMENT_CONFIG, indent=2))

sample_loss_fn = tf.keras.losses.CategoricalCrossentropy(reduction="none")   # one loss per image

history = pd.read_csv(HISTORY_PATH) if HISTORY_PATH.is_file() else pd.DataFrame(columns=list(HISTORY_DTYPES))
history = history.astype(HISTORY_DTYPES)      # keep numeric dtypes even when the run starts from an empty history
print("\nepochs already in history:", len(history))

def evaluate(model, dataset):
    """Unweighted loss, accuracy and macro F1 over a tf.data dataset (validation and test only)."""
    total_loss, seen, true_labels, predictions = 0.0, 0, [], []
    for images, labels in dataset:
        probabilities = model(images, training=False)
        total_loss += float(tf.reduce_sum(sample_loss_fn(labels, probabilities)))
        true_labels.append(np.argmax(labels.numpy(), axis=1))
        predictions.append(np.argmax(probabilities.numpy(), axis=1))
        seen += int(images.shape[0])
    true_labels, predictions = np.concatenate(true_labels), np.concatenate(predictions)
    return (total_loss / seen, float(np.mean(true_labels == predictions)),
            float(sklearn.metrics.f1_score(true_labels, predictions, average="macro")))

def make_train_step(optimizer):
    """One graph-mode step: categorical cross-entropy, gradient norm and accuracy."""
    @tf.function
    def train_step(images, labels):
        with tf.GradientTape() as tape:
            probabilities = model(images, training=True)
            loss = tf.reduce_mean(sample_loss_fn(labels, probabilities))
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        accuracy = tf.reduce_mean(tf.cast(tf.argmax(probabilities, 1) == tf.argmax(labels, 1), tf.float32))
        return loss, tf.linalg.global_norm(gradients), accuracy
    return train_step

def cosine_learning_rate(epoch_in_stage):
    """Cosine decay from FINETUNE_LR_START to FINETUNE_LR_END across the fine-tuning stage."""
    position = epoch_in_stage / max(1, STAGE_B_EPOCHS - 1)
    return float(FINETUNE_LR_END + 0.5 * (FINETUNE_LR_START - FINETUNE_LR_END) * (1 + math.cos(math.pi * position)))

best = {"val_acc": -1.0, "epoch": 0, "val_loss": float("nan")}
if len(history):
    best_row = history.loc[history["val_acc"].idxmax()]        # idxmax keeps the earlier epoch on a tie
    best = {"val_acc": float(best_row["val_acc"]), "epoch": int(best_row["epoch"]),
            "val_loss": float(best_row["val_loss"])}

def run_stage(stage, optimizer, epochs, first_epoch, lr_of_epoch):
    """Run `epochs` epochs from `first_epoch`; checkpoint every epoch and keep the best validation accuracy."""
    global history, best
    train_step = make_train_step(optimizer)
    print(f"\nstage {stage}: {epochs} epoch(s), batch {BATCH_SIZE}, trainable weights "
          f"{sum(int(np.prod(weight.shape)) for weight in model.trainable_weights):,}")
    for offset in range(epochs):
        epoch = first_epoch + offset
        learning_rate = lr_of_epoch(offset)
        optimizer.learning_rate.assign(learning_rate)
        start = time.perf_counter()
        losses, accuracies, gradient_norms = [], [], []
        for images, labels in train_ds:
            loss_value, gradient_norm, accuracy = train_step(images, labels)
            losses.append(float(loss_value))
            gradient_norms.append(float(gradient_norm))
            accuracies.append(float(accuracy))
        val_loss, val_accuracy, val_macro_f1 = evaluate(model, val_ds)
        row = {"stage": stage, "epoch": epoch, "lr": learning_rate,
               "train_loss": float(np.mean(losses)), "train_acc": float(np.mean(accuracies)),
               "val_loss": val_loss, "val_acc": val_accuracy, "val_macro_f1": val_macro_f1,
               "grad_norm": float(np.mean(gradient_norms)), "seconds": time.perf_counter() - start}
        history = pd.concat([history, pd.DataFrame([row])], ignore_index=True)
        history.to_csv(HISTORY_PATH, index=False)
        model.save_weights(LAST_WEIGHTS)                       # resume point for an interrupted run
        if val_accuracy > best["val_acc"]:                     # strict comparison keeps the earlier epoch on a tie
            best = {"val_acc": val_accuracy, "epoch": epoch, "val_loss": val_loss}
            model.save_weights(BEST_WEIGHTS)
        print("  epoch {epoch:3d} | train loss {train_loss:.4f} acc {train_acc:.3f} | "
              "val loss {val_loss:.4f} acc {val_acc:.3f} f1 {val_macro_f1:.3f} | lr {lr:.1e} | "
              "grad norm {grad_norm:.2f} | {seconds:.0f}s".format(**row), flush=True)

done_a = int((history["stage"] == "A").sum())
done_b = int((history["stage"] == "B").sum())
if done_a + done_b < TOTAL_EPOCHS:
    if LAST_WEIGHTS.is_file():
        model.load_weights(LAST_WEIGHTS)
        print("resumed weights from", LAST_WEIGHTS)
    if done_a < STAGE_A_EPOCHS:
        base_model.trainable = False
        for layer in base_model.layers:
            layer.trainable = False
        run_stage("A", tf.keras.optimizers.Adam(HEAD_LR), STAGE_A_EPOCHS - done_a,
                  first_epoch=len(history) + 1, lr_of_epoch=lambda offset: HEAD_LR)
    if done_b < STAGE_B_EPOCHS:
        base_model.trainable = True
        for layer in base_model.layers:
            conv_block = layer.name.startswith(("conv4", "conv5"))
            layer.trainable = conv_block and not isinstance(layer, tf.keras.layers.BatchNormalization)
        run_stage("B", tf.keras.optimizers.Adam(FINETUNE_LR_START), STAGE_B_EPOCHS - done_b,
                  first_epoch=len(history) + 1,
                  lr_of_epoch=lambda offset: cosine_learning_rate(done_b + offset))
else:
    print("training already complete: reusing", HISTORY_PATH)

assert len(history) == TOTAL_EPOCHS, f"history holds {len(history)} epochs, expected {TOTAL_EPOCHS}"
assert history["epoch"].tolist() == list(range(1, TOTAL_EPOCHS + 1)), "epoch numbering must be continuous"
assert np.isfinite(history[["train_loss", "val_loss", "train_acc", "val_acc"]].to_numpy()).all()

# 2.11 - The selected checkpoint must reproduce the validation accuracy it was selected for.
model.load_weights(BEST_WEIGHTS)
restored_loss, restored_accuracy, restored_macro_f1 = evaluate(model, val_ds)
assert np.isclose(restored_accuracy, best["val_acc"], atol=1e-6), \
    "the restored checkpoint does not reproduce the selected validation accuracy"
print(f"\nselected epoch {best['epoch']} | val acc {restored_accuracy:.4f} | val loss {restored_loss:.4f} "
      f"| val macro F1 {restored_macro_f1:.4f}")


## 2.12–2.13 Learning curves and overfitting analysis

The loss curves are drawn from `outputs/tables/train_history.csv`, with the learning-rate schedule on the second axis and vertical lines for the stage boundary and the selected epoch. The second figure plots validation macro F1 per epoch, because accuracy alone hides class-wise regressions on this imbalanced split. The analysis cell derives every number quoted in section 2.13 from the same table and writes it to `outputs/tables/curve_analysis.txt`, so the prose and the recorded evidence cannot drift apart after a rerun.


In [ ]:
# 2.12 - Training curves: loss with the learning-rate schedule on a second axis.
epochs_axis = history["epoch"].to_numpy()
fig, ax_loss = plt.subplots(figsize=(11, 5))
ax_loss.plot(epochs_axis, history["train_loss"], color="tab:blue", marker="o", markersize=2.5,
             linewidth=1, label="training loss")
ax_loss.plot(epochs_axis, history["val_loss"], color="tab:red", marker="o", markersize=2.5,
             linewidth=1, label="validation loss")
ax_loss.axvline(STAGE_A_EPOCHS + 0.5, color="grey", linestyle="--", alpha=0.7, label="fine-tuning starts")
ax_loss.axvline(best["epoch"], color="tab:red", linestyle=":", alpha=0.8, label=f"selected epoch {best['epoch']}")
ax_loss.set_xlabel("epoch")
ax_loss.set_ylabel("categorical cross-entropy")
ax_loss.grid(alpha=0.3)
ax_lr = ax_loss.twinx()
ax_lr.plot(epochs_axis, history["lr"], color="tab:green", linestyle="--", linewidth=1, label="learning rate")
ax_lr.set_yscale("log")
ax_lr.set_ylabel("learning rate (log scale)")
handles = ax_loss.get_lines() + ax_lr.get_lines()
ax_loss.legend(handles, [handle.get_label() for handle in handles], loc="upper right", fontsize=8)
ax_loss.set_title(f"2.12  training / validation loss and learning-rate schedule ({len(history)} epochs)")
fig.tight_layout()
fig.savefig(OUT_DIR / "figures" / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# The validation macro F1 curve shows whether the accuracy-selected epoch is also the balanced-metric
# epoch. Selection still uses validation accuracy (recorded in the experiment configuration).
fig, ax_f1 = plt.subplots(figsize=(11, 3.5))
ax_f1.plot(epochs_axis, history["val_macro_f1"], color="tab:purple", marker="o", markersize=2.5,
           linewidth=1, label="validation macro F1")
ax_f1.axvline(best["epoch"], color="tab:red", linestyle=":", alpha=0.8, label=f"selected epoch {best['epoch']}")
ax_f1.set_xlabel("epoch")
ax_f1.set_ylabel("macro F1")
ax_f1.grid(alpha=0.3)
ax_f1.legend(fontsize=8)
ax_f1.set_title("2.12  validation macro F1 per epoch")
fig.tight_layout()
fig.savefig(OUT_DIR / "figures" / "validation_macro_f1.png", dpi=150, bbox_inches="tight")
plt.show()

# 2.12 - Gradient-norm evidence: the frozen stage has fewer trainable tensors, so the top rows of the
# table must be visibly smaller than the fine-tuning rows.
stage_summary = history.groupby("stage")["grad_norm"].agg(["mean", "min", "max"]).round(2)
print(stage_summary.to_string())
print("epochs with the lowest validation loss:", history.nsmallest(3, "val_loss")[["epoch", "stage", "val_loss", "val_acc"]].to_string(index=False))


In [ ]:
# 2.13 - Overfitting evidence derived from the saved history (also written to a text file).
selected_row = history[history["epoch"] == best["epoch"]].iloc[0]
last_row = history.iloc[-1]
tail = history.tail(10)
lowest_val_loss_row = history.loc[history["val_loss"].idxmin()]
best_f1_row = history.loc[history["val_macro_f1"].idxmax()]
warning_windows = []
for index in range(2, len(history)):
    window = history.iloc[index - 2:index + 1]
    if (window["stage"].nunique() == 1 and (np.diff(window["val_loss"]) > 0).all()
            and (np.diff(window["train_loss"]) < 0).all()):
        warning_windows.append((int(window.iloc[0]["epoch"]), int(window.iloc[-1]["epoch"])))
stage_a_grad = float(history.loc[history["stage"] == "A", "grad_norm"].mean())
stage_b_grad = float(history.loc[history["stage"] == "B", "grad_norm"].mean())

analysis = "\n".join([
    f"Epochs completed: {len(history)} ({STAGE_A_EPOCHS} head + {STAGE_B_EPOCHS} fine-tuning); "
    f"total training time {history['seconds'].sum() / 60:.1f} min.",
    f"Selected epoch: {best['epoch']} (highest validation accuracy {selected_row['val_acc']:.4f}, "
    f"validation loss {selected_row['val_loss']:.4f}, validation macro F1 {selected_row['val_macro_f1']:.4f}). "
    "That checkpoint is restored before every test number below.",
    f"Lowest validation loss: {lowest_val_loss_row['val_loss']:.4f} at epoch {int(lowest_val_loss_row['epoch'])}; "
    f"highest validation macro F1: {best_f1_row['val_macro_f1']:.4f} at epoch {int(best_f1_row['epoch'])}. "
    "Selecting by accuracy therefore " + ("agrees with" if int(lowest_val_loss_row["epoch"]) == best["epoch"] else "disagrees with")
    + " the lowest-loss epoch.",
    f"Final epoch: train loss {last_row['train_loss']:.4f}, train accuracy {last_row['train_acc']:.3f}, "
    f"validation loss {last_row['val_loss']:.4f}, validation accuracy {last_row['val_acc']:.3f}, "
    f"validation macro F1 {last_row['val_macro_f1']:.4f}; train/validation loss gap "
    f"{last_row['val_loss'] - last_row['train_loss']:+.4f} nat.",
    f"Last ten epochs: validation accuracy range {100 * (tail['val_acc'].max() - tail['val_acc'].min()):.2f} "
    f"percentage points, validation loss range {tail['val_loss'].max() - tail['val_loss'].min():.4f}.",
    "First window where validation loss rises on two consecutive epochs while training loss falls: "
    f"{warning_windows[0] if warning_windows else 'none'}; this is a descriptive warning, not proof of an exact onset.",
    f"Mean gradient norm: {stage_a_grad:.2f} with the frozen backbone, {stage_b_grad:.2f} after unfreezing conv4/conv5.",
])
print(analysis)
(OUT_DIR / "tables" / "curve_analysis.txt").write_text(analysis + "\n")


## 2.13 Learning-curve analysis

RESULT_PLACEHOLDER_2_13


## 2.14–2.17 Internal test evaluation

Required: test accuracy and loss, normalised confusion matrix, top-3 confident errors, error analysis. Produces: evaluation figures and tables.

**Internal test.** One prediction pass over the held-out lesion-disjoint test split supplies every number in this section: loss, accuracy, balanced accuracy, macro F1, the row-normalised 7x7 confusion matrix and the three most confident mistakes. Per-class precision/recall and the most frequent confusion pairs are printed and saved, and they are the evidence for the error analysis in section 2.17.

RESULT_PLACEHOLDER_2_14


In [ ]:
# 2.14 - Internal test evaluation with the selected checkpoint. One prediction pass supplies every metric.
model.load_weights(BEST_WEIGHTS)
test_probs, test_labels = [], []
for images, labels in test_ds:
    test_probs.append(model(images, training=False).numpy())
    test_labels.append(labels.numpy().argmax(axis=1))
test_probs = np.concatenate(test_probs)
test_labels = np.concatenate(test_labels)
test_rows = isic2018[isic2018["split"] == "test"].reset_index(drop=True)
assert test_probs.shape == (len(test_rows), len(CLASS_NAMES)), "prediction count does not match the test split"
np.testing.assert_allclose(test_probs.sum(axis=1), 1.0, atol=1e-5, err_msg="test probabilities do not sum to one")
assert np.array_equal(test_labels, np.array([CLASS_TO_INDEX[label] for label in test_rows["label"]]))

test_pred = test_probs.argmax(axis=1)
test_confidence = test_probs.max(axis=1)
test_loss = float(np.mean(sample_loss_fn(tf.one_hot(test_labels, len(CLASS_NAMES)),
                                          tf.constant(test_probs, dtype=tf.float32)).numpy()))
test_accuracy = float(np.mean(test_pred == test_labels))
test_macro_f1 = float(sklearn.metrics.f1_score(test_labels, test_pred, average="macro"))
test_balanced_accuracy = float(sklearn.metrics.balanced_accuracy_score(test_labels, test_pred))
print(f"internal test ({len(test_rows)} images, selected epoch {best['epoch']}): loss {test_loss:.4f} | "
      f"accuracy {test_accuracy:.4f} | balanced accuracy {test_balanced_accuracy:.4f} | "
      f"macro F1 {test_macro_f1:.4f}")

classification_report = sklearn.metrics.classification_report(
    test_labels, test_pred, target_names=CLASS_NAMES, digits=4, zero_division=0)
(OUT_DIR / "tables" / "test_classification_report.txt").write_text(classification_report + "\n")
print(classification_report)

# 2.15 - Row-normalised confusion matrix (each row sums to 1).
confusion = sklearn.metrics.confusion_matrix(test_labels, test_pred, labels=np.arange(len(CLASS_NAMES)))
confusion_normalised = confusion / confusion.sum(axis=1, keepdims=True)
np.testing.assert_allclose(confusion_normalised.sum(axis=1), 1.0)
fig, ax = plt.subplots(figsize=(8, 7))
image_handle = ax.imshow(confusion_normalised, cmap="Blues", vmin=0, vmax=1)
for row in range(len(CLASS_NAMES)):
    for column in range(len(CLASS_NAMES)):
        value = confusion_normalised[row, column]
        ax.text(column, row, f"{value:.2f}", ha="center", va="center", fontsize=8,
                color="white" if value > 0.5 else "black")
ax.set_xticks(range(len(CLASS_NAMES)))
ax.set_yticks(range(len(CLASS_NAMES)))
ax.set_xticklabels(CLASS_NAMES, rotation=45)
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("predicted class")
ax.set_ylabel("true class")
ax.set_title("2.15  row-normalised confusion matrix (internal test)")
fig.colorbar(image_handle, ax=ax, fraction=0.046)
fig.tight_layout()
fig.savefig(OUT_DIR / "figures" / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# 2.16 - The three most confident wrong predictions, chosen by predicted-class probability.
wrong_indices = np.where(test_pred != test_labels)[0]
assert len(wrong_indices) > 0, "no wrong test predictions: the error analysis would be meaningless"
confident_wrong = wrong_indices[np.argsort(test_confidence[wrong_indices])[::-1][:3]]
fig, axes = plt.subplots(1, len(confident_wrong), figsize=(4.5 * len(confident_wrong), 4.5), squeeze=False)
for axis, index in zip(axes.flat, confident_wrong):
    axis.imshow(Image.open(test_rows.loc[index, "path"]).convert("RGB"))
    axis.set_title(f"true: {CLASS_NAMES[test_labels[index]]}\n"
                   f"pred: {CLASS_NAMES[test_pred[index]]} ({test_confidence[index]:.3f})", fontsize=9)
    axis.axis("off")
fig.suptitle("2.16  top-3 most confident wrong predictions (internal test)", fontsize=11)
fig.tight_layout()
fig.savefig(OUT_DIR / "figures" / "top_confident_errors.png", dpi=150, bbox_inches="tight")
plt.show()
print("most confident errors:", [
    (CLASS_NAMES[test_labels[index]], CLASS_NAMES[test_pred[index]],
     round(float(test_confidence[index]), 3)) for index in confident_wrong])

# 2.17 - Error analysis evidence: most frequent confusion pairs.
confusion_pairs = [{"true": CLASS_NAMES[true_index], "predicted": CLASS_NAMES[predicted_index],
                    "count": int(confusion[true_index, predicted_index]),
                    "row_fraction": float(confusion_normalised[true_index, predicted_index])}
                   for true_index in range(len(CLASS_NAMES)) for predicted_index in range(len(CLASS_NAMES))
                   if true_index != predicted_index and confusion[true_index, predicted_index] > 0]
confusion_pairs = pd.DataFrame(confusion_pairs).sort_values("count", ascending=False)
confusion_pairs.to_csv(OUT_DIR / "tables" / "test_confusion_pairs.csv", index=False)
print(confusion_pairs.head(6).to_string(index=False))

# Full prediction table for the report appendix.
predictions = test_rows[["image", "lesion_id", "path", "label"]].copy()
predictions["true_class"] = [CLASS_NAMES[index] for index in test_labels]
predictions["predicted_class"] = [CLASS_NAMES[index] for index in test_pred]
predictions["confidence"] = test_confidence
for index, name in enumerate(CLASS_NAMES):
    predictions["p_" + name] = test_probs[:, index]
assert len(predictions) == len(test_rows)
predictions.to_csv(OUT_DIR / "tables" / "test_predictions.csv", index=False)

test_metrics = {"test_loss": test_loss, "test_accuracy": test_accuracy,
                "balanced_accuracy": test_balanced_accuracy, "macro_f1": test_macro_f1,
                "test_images": len(test_rows), "selected_epoch": int(best["epoch"]),
                "epochs_completed": int(len(history))}
(OUT_DIR / "tables" / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))


## 2.18–2.20 External dataset, unseen image and calibration

Required: external evaluation, unseen-image probabilities, softmax discussion, calibration. Produces: external table, bar chart, reliability diagram and ECE.

**Softmax and calibration.** The dense head produces logits $z$; softmax maps them to $p_k = \exp(z_k - m) / \sum_j \exp(z_j - m)$ with $m = \max_j z_j$, so the seven outputs are positive, sum to one and cannot overflow. A classifier is calibrated when predictions made at 80% confidence are correct about 80% of the time. The reliability diagram bins test predictions by confidence and compares mean confidence with empirical accuracy; the expected calibration error (ECE) is the count-weighted average gap between the two.

RESULT_PLACEHOLDER_2_18


In [ ]:
# 2.18 - External evaluation on ISIC 2019 (7-class mapping, hash-deduplicated against ISIC 2018).
model.load_weights(BEST_WEIGHTS)

def external_dataset(dataframe, batch_size=BATCH_SIZE):
    paths = tf.constant(dataframe["path"].tolist())
    labels = tf.constant([CLASS_TO_INDEX[label] for label in dataframe["label"]], dtype=tf.int32)
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels)).map(
        lambda path, label: (tf.cast(decode_resize(path), tf.float32), tf.one_hot(label, len(CLASS_NAMES))),
        num_parallel_calls=AUTOTUNE)
    return dataset.batch(batch_size).prefetch(AUTOTUNE)

start = time.perf_counter()
external_probs, external_loss_sum, external_seen = [], 0.0, 0
for images, labels in external_dataset(isic2019):
    probabilities = model(images, training=False)
    external_probs.append(probabilities.numpy())
    external_loss_sum += float(tf.reduce_sum(sample_loss_fn(labels, probabilities)))
    external_seen += int(images.shape[0])
external_probs = np.concatenate(external_probs)
external_labels = np.array([CLASS_TO_INDEX[label] for label in isic2019["label"]])
external_pred = external_probs.argmax(axis=1)
external_confidence = external_probs.max(axis=1)
assert external_seen == len(isic2019), "external pass did not cover every image"
np.testing.assert_allclose(external_probs.sum(axis=1), 1.0, atol=1e-5)
external_loss = external_loss_sum / external_seen
external_accuracy = float((external_pred == external_labels).mean())
external_balanced_accuracy = float(sklearn.metrics.balanced_accuracy_score(external_labels, external_pred))
external_macro_f1 = float(sklearn.metrics.f1_score(external_labels, external_pred, average="macro"))
print(f"ISIC 2019 external ({external_seen} images): loss {external_loss:.4f} | accuracy {external_accuracy:.4f} | "
      f"balanced accuracy {external_balanced_accuracy:.4f} | macro F1 {external_macro_f1:.4f} | "
      f"{time.perf_counter() - start:.0f}s")

external_table = pd.DataFrame({
    "class": CLASS_NAMES,
    "support": [int((external_labels == index).sum()) for index in range(len(CLASS_NAMES))],
    "predicted": [int((external_pred == index).sum()) for index in range(len(CLASS_NAMES))],
    "recall": [float(((external_pred == index) & (external_labels == index)).sum()
                     / max((external_labels == index).sum(), 1)) for index in range(len(CLASS_NAMES))],
})
assert external_table["support"].sum() == external_seen
external_table.to_csv(OUT_DIR / "tables" / "external_isic2019_per_class.csv", index=False)
print(external_table.to_string(index=False))

# 2.19 - One reusable single-image function that applies exactly the evaluation preprocessing, plus one
# fixed unseen ISIC 2019 image shown as a probability bar chart.
def predict_image(image_path):
    """Return the seven class probabilities for one JPEG, using the selected model."""
    image = tf.cast(decode_resize(image_path), tf.float32)
    return model(image[None], training=False).numpy()[0]

# The single-image path must reproduce the batched test prediction, otherwise every later example is wrong.
reference_path = isic2018[isic2018["split"] == "test"].reset_index(drop=True).loc[0, "path"]
np.testing.assert_allclose(predict_image(reference_path), test_probs[0], rtol=1e-4, atol=1e-5)
print("single-image preprocessing matches the batched test prediction")

unseen_row = isic2019.iloc[0]
unseen_probs = predict_image(unseen_row["path"])
unseen_image = np.asarray(Image.open(unseen_row["path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32) / 255.0
malignant_probability = float(sum(unseen_probs[CLASS_TO_INDEX[label]] for label in ("MEL", "BCC", "AKIEC")))
fig, (image_axis, bar_axis) = plt.subplots(1, 2, figsize=(12, 4.5), gridspec_kw={"width_ratios": [1, 1.4]})
image_axis.imshow(unseen_image)
image_axis.set_title(f"unseen image {unseen_row['image']}\ntrue class: {unseen_row['label']}", fontsize=9)
image_axis.axis("off")
bar_axis.barh(CLASS_NAMES[::-1], unseen_probs[::-1], color="tab:blue")
for index, probability in enumerate(unseen_probs[::-1]):
    bar_axis.text(probability + 0.01, index, f"{probability:.3f}", va="center", fontsize=8)
bar_axis.set_xlim(0, 1.05)
bar_axis.set_xlabel("softmax probability")
bar_axis.set_title(f"predicted {CLASS_NAMES[int(unseen_probs.argmax())]} | "
                   f"P(malignant) = {malignant_probability:.3f}", fontsize=10)
fig.suptitle("2.19  probabilities for one unseen ISIC 2019 image", fontsize=11)
fig.tight_layout()
fig.savefig(OUT_DIR / "figures" / "unseen_image_probabilities.png", dpi=150, bbox_inches="tight")
plt.show()
print("unseen image:", unseen_row["image"], "| true:", unseen_row["label"],
      "| predicted:", CLASS_NAMES[int(unseen_probs.argmax())], "| P(malignant):", round(malignant_probability, 3))

# 2.20 - Reliability diagram and expected calibration error on the internal test set.
def calibration_table(probabilities, labels, bins=10):
    confidence = probabilities.max(axis=1)
    correct = (probabilities.argmax(axis=1) == labels).astype(float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    rows = []
    for lower, upper in zip(edges[:-1], edges[1:]):
        mask = (confidence > lower) & (confidence <= upper)
        if mask.sum() == 0:
            continue
        rows.append({"bin_center": (lower + upper) / 2, "count": int(mask.sum()),
                     "mean_confidence": float(confidence[mask].mean()),
                     "accuracy": float(correct[mask].mean())})
    table = pd.DataFrame(rows)
    ece = float(np.sum(table["count"] / len(labels) * (table["accuracy"] - table["mean_confidence"]).abs()))
    return table, ece

reliability, ece = calibration_table(test_probs, test_labels)
assert 0.0 <= ece <= 1.0
reliability.to_csv(OUT_DIR / "tables" / "calibration_reliability.csv", index=False)
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.bar(reliability["mean_confidence"], reliability["accuracy"], width=0.08,
       color="tab:blue", alpha=0.6, label="observed accuracy")
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="perfect calibration")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("mean predicted confidence")
ax.set_ylabel("empirical accuracy")
ax.set_title(f"2.20  reliability diagram (ECE = {ece:.4f})")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "figures" / "reliability_diagram.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"expected calibration error (internal test): {ece:.4f}")
print("mean confidence on wrong predictions: %.4f | mean confidence on correct predictions: %.4f" % (
    test_confidence[test_labels != test_pred].mean(),
    test_confidence[test_labels == test_pred].mean()))

external_metrics = {"images": int(external_seen), "loss": external_loss, "accuracy": external_accuracy,
                    "balanced_accuracy": external_balanced_accuracy, "macro_f1": external_macro_f1,
                    "mean_confidence": float(external_confidence.mean()),
                    "calibration_ece_internal_test": ece}
(OUT_DIR / "tables" / "external_isic2019_metrics.json").write_text(json.dumps(external_metrics, indent=2))


## Artifacts, reproducibility and submission packaging

Required: one self-contained notebook and a verified results archive. Produces: weights, metrics JSON, all figures, results ZIP.

**Artifacts.** Everything the report needs is written with fixed names under `outputs/`:
`figures/` (augmentation, feature maps, gradient magnitudes, training curves, validation macro F1, confusion matrix, confident errors, unseen-image probabilities, reliability diagram), `tables/` (environment, split manifest, data provenance, experiment configuration, model summary, training history, curve analysis, classification report, test predictions, metrics, calibration table) and `model/` (best and last weights). `outputs/Group_8_Results.zip` bundles figures, tables and the selected weights, so the results can be checked without rerunning the 100-epoch training.


In [ ]:
# Artifacts, reproducibility and submission packaging.
import zipfile

metrics = {
    "experiment_config": EXPERIMENT_CONFIG,
    "environment": ENV,
    "seed": SEED,
    "splits": {name: int((isic2018["split"] == name).sum()) for name in ("train", "validation", "test")},
    "internal_test": json.loads((OUT_DIR / "tables" / "test_metrics.json").read_text()),
    "external_isic2019": json.loads((OUT_DIR / "tables" / "external_isic2019_metrics.json").read_text()),
    "selected_epoch": int(best["epoch"]),
    "selected_val_accuracy": float(best["val_acc"]),
    "curve_analysis": (OUT_DIR / "tables" / "curve_analysis.txt").read_text(),
    "external_rows": int(len(isic2019)),
}
assert metrics["splits"]["train"] + metrics["splits"]["validation"] + metrics["splits"]["test"] == 10015
(OUT_DIR / "tables" / "metrics.json").write_text(json.dumps(metrics, indent=2))

archive_path = OUT_DIR / "Group_8_Results.zip"
if archive_path.is_file():
    archive_path.unlink()
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in ("figures", "tables"):
        for file_path in sorted((OUT_DIR / folder).rglob("*")):
            if file_path.is_file():
                archive.write(file_path, file_path.relative_to(OUT_DIR))
    archive.write(BEST_WEIGHTS, "model/best.weights.h5")      # selected model only
with zipfile.ZipFile(archive_path) as archive:
    names = sorted(archive.namelist())
required = {"tables/experiment_config.json", "tables/data_provenance.json", "tables/metrics.json",
            "tables/train_history.csv", "tables/curve_analysis.txt", "model/best.weights.h5",
            "figures/training_curves.png", "figures/confusion_matrix.png", "figures/reliability_diagram.png"}
assert required.issubset(set(names)), f"missing artifacts: {sorted(required - set(names))}"
print(f"results archive: {archive_path} ({archive_path.stat().st_size / 1024 / 1024:.1f} MB, {len(names)} files)")
print("\n".join(names))
